## 주요 변경점 및 결과
1. `Regularization_v2_8.ipynb`를 복사한 형식을 유지하면서 `src/models`의 아키텍처를 순회 비교합니다.
2. 비교 대상: `MultiViewFeatureFusion`, `MultiViewCrossAttention`, `MultiViewBidirectionalCrossAttention`, `MultiViewDualEncoderBidirectionalCrossAttention`.
3. dev 기준 `logloss`, `accuracy`, `AUC`, TTA 결과, 학습 곡선을 함께 저장합니다.


In [ ]:
import os
os.environ['CUDA_MPS_PIPE_DIRECTORY'] = '/tmp/nvidia-mps'
os.environ['CUDA_MPS_LOG_DIRECTORY'] = '/tmp/nvidia-mps-log'

## 1. 라이브러리 및 환경 설정

In [ ]:
import os
import sys
import json
import random
import pandas as pd
import numpy as np
import cv2
import shutil
import timm
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from pathlib import Path
from dataclasses import dataclass, asdict

SRC_DIR = (Path.cwd() / '../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from augmentations import build_default_transforms
from output_paths import allocate_output_paths
from reproducibility import make_generator, seed_everything, seed_worker

# /src 에서 실행하는 기준 경로 설정
DATA_DIR = (Path.cwd() / '../data').resolve()
assert DATA_DIR.exists(), f"data 폴더를 찾지 못했습니다: {DATA_DIR}"
print(f"DATA_DIR: {DATA_DIR}")

# 하이퍼파라미터 설정
CFG = {
    # Basic
    'IMG_SIZE': 320,
    'EPOCHS': [30, 50, 100],
    'LEARNING_RATE': 3e-4,
    'MIN_LR': 1e-6,
    'BATCH_SIZE': 32,
    'SEED': 42,
    'NUM_WORKERS': 16,
    'USE_AMP' : True,
    # Model comparison
    'MODEL_NAMES': [
        'feature_fusion',
        'cross_attention',
        'bidirectional_cross_attention',
        'dual_encoder_bidirectional_cross_attention',
    ],
    'BACKBONE_NAME': 'efficientnetv2_rw_s',
    'FRONT_BACKBONE_NAME': None,
    'TOP_BACKBONE_NAME': None,
    'PRETRAINED': True,
    'ATTN_DIM': 256,
    'NUM_HEADS': 8,
    'NUM_LAYERS': 2,
    'ATTN_DROPOUT': 0.1,
    'CLASSIFIER_DROPOUT': 0.3,
    'USE_WANDB': False,
    'PORTFOLIO_ARTIFACTS': True,
    # Regularization & Stability
    'WEIGHT_DECAY': 1e-4,  # L2 regularization
    'EMA_DECAY': 0.999, # 시계열에서 window size만큼 고려해 지역적 평균 구하는 방식으로 노이즈를 제거
    'EMA_USE_FOR_EVAL': True,
    'PATIENCE' : 10,
    # Augmentation & TTA
    'TTA_CANDIDATES': [ # TEST TIME AUGMENTATION
        ['none'],
        ['none', 'hflip'],
        ['none', 'hflip', 'crop95'],
    ],
    'DISTILL_WEIGHT': 0.0, # 모델 비교에서는 VideoMAE distillation을 사용하지 않음
    # video frame augmentation (for unstable videos)
    'VIDEO_AUG_ENABLE': False,
    'VIDEO_AUG_CACHE': True,
    'UNSTABLE_START_MIN_SEC': 0.5,
    'UNSTABLE_START_MAX_SEC': 1.0,
    'UNSTABLE_FRAMES_MIN': 2,
    'UNSTABLE_FRAMES_MAX': 3,
    'STABLE_END_MIN_SEC': 9.0,
    'STABLE_END_MAX_SEC': 10.0,
    'STABLE_FRAMES_PER_VIDEO': 2,
}

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## 2. 데이터 로드 및 학습/검증 데이터 분할

In [ ]:
# 1. 데이터 로드
train_df = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8-sig')
val_df = pd.read_csv(DATA_DIR / 'dev.csv', encoding='utf-8-sig')

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")

## 3. 영상에서 Feature을 추출하는 Teacher 모델 선언

원본 `Regularization_v2_8.ipynb`의 형식은 유지하지만, 이번 노트북은 `src/models` 아키텍처 비교가 목적이므로 VideoMAE teacher feature 추출과 distillation은 사용하지 않습니다.


In [ ]:
# Model comparison에서는 VideoMAE teacher feature 추출을 생략합니다.
# 원본 Regularization 노트북과 데이터셋 인터페이스를 맞추기 위해 FEATURE_DIR 변수만 유지합니다.
FEATURE_DIR = DATA_DIR / 'videomae_features'
print(f"FEATURE_DIR(optional): {FEATURE_DIR}")
print("Distillation: disabled for src/models architecture comparison")


In [ ]:
from PIL import Image
from pathlib import Path

# 1. 샘플 이미지 경로 하나 지정 (앞서 사용하신 경로 기준)
sample_path = Path(DATA_DIR) / "train" / "TRAIN_0010" / "front.png" 

# 2. 이미지 열기
img = Image.open(sample_path)

# 3. 원본 사이즈 확인 (가로, 세로)
width, height = img.size
print(f"원본 이미지 사이즈: {width} x {height}")

In [ ]:
print("VideoMAE hidden dimension check skipped: distillation is disabled in this comparison notebook.")


모델 비교 노트북에서는 VideoMAE teacher feature를 사용하지 않습니다.

이 섹션은 `Regularization_v2_8.ipynb`의 형식을 유지하기 위해 남겨두었고, 실제 학습은 이미지 기반 `src/models` 아키텍처만 비교합니다.


In [ ]:
# 원본 Regularization 노트북에서는 이 셀에서 VideoMAE feature를 추출했습니다.
# 모델 아키텍처 비교에서는 teacher feature가 필요 없으므로 생략합니다.
# 기존에 추출된 feature가 있더라도 train_one_epoch에서는 사용하지 않습니다.
print("Skip VideoMAE feature extraction for model comparison.")


## 4-1. CheckerboardTopNormalizer 클래스 & CheckerboardTopNormConfig 설정 정의

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional
from PIL import Image

import cv2
import numpy as np
import matplotlib.pyplot as plt

@dataclass(frozen=True)
class CheckerboardTopNormConfig:
    enabled: bool = True
    ring_ratio: float = 0.10
    rot_line_min: int = 10
    rot_conf_min: float = 0.20
    pad_value: int = 128


def _estimate_mask(rgb: np.ndarray) -> np.ndarray:
    """
    지저분한 마스크 다듬기 & 원하는 물체만 골라내기
    """
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY) 
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV) 
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]

    s_thr = float(np.percentile(sat, 60.0))
    g_thr = float(np.percentile(gray, 45.0))
    v_thr = float(np.percentile(val, 35.0))

    mask = ((sat > s_thr) | (gray < g_thr) | (val < v_thr)).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8), iterations=2)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n <= 1:
        return (mask > 0).astype(np.uint8)

    best = 1
    best_area = 0
    h, w = gray.shape
    for i in range(1, n):
        x, y, ww, hh, area = stats[i].tolist()
        if area > best_area and ww > 8 and hh > 8 and area < 0.995 * h * w:
            best = i
            best_area = area
    return (labels == best).astype(np.uint8)


def _ring_mask(h: int, w: int, ratio: float) -> np.ndarray:
    r = max(1, int(round(min(h, w) * ratio)))
    mask = np.zeros((h, w), dtype=np.uint8)
    mask[:r, :] = 1
    mask[-r:, :] = 1
    mask[:, :r] = 1
    mask[:, -r:] = 1
    return mask


def _line_angles(edge: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    추출한 Edge 이미지에서 직선을 찾고, 기울어진 각을 계산해 반환하는 함수
    """
    lines = cv2.HoughLinesP(edge, 1, np.pi / 180.0, threshold=30, minLineLength=24, maxLineGap=6)
    if lines is None or len(lines) == 0:
        return np.zeros((0,), dtype=np.float32), np.zeros((0,), dtype=np.float32)

    angs = []
    lens = []
    for line in lines[:400]:
        x1, y1, x2, y2 = line[0].tolist()
        dx = float(x2 - x1)
        dy = float(y2 - y1)
        ln = float(np.hypot(dx, dy))
        if ln < 8:
            continue
        ang = (float(np.degrees(np.arctan2(dy, dx))) + 180.0) % 180.0
        angs.append(ang)
        lens.append(ln)
    if len(angs) == 0:
        return np.zeros((0,), dtype=np.float32), np.zeros((0,), dtype=np.float32)
    return np.asarray(angs, dtype=np.float32), np.asarray(lens, dtype=np.float32)


def estimate_top_rotation(rgb: np.ndarray, cfg: CheckerboardTopNormConfig) -> dict[str, float | bool | int | str]:
    h, w = rgb.shape[:2]
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    fg = _estimate_mask(rgb) # 원하는 물체를 추출하는 함수
    bg = (1 - fg).astype(np.uint8)
    if int(bg.sum()) < int(0.03 * h * w):
        bg = _ring_mask(h, w, cfg.ring_ratio)

    gx = cv2.Scharr(gray, cv2.CV_32F, 1, 0)
    gy = cv2.Scharr(gray, cv2.CV_32F, 0, 1)
    mag = cv2.magnitude(gx, gy)
    mag = (mag / (mag.max() + 1e-6) * 255.0).astype(np.uint8)
    edges = cv2.Canny(mag, 40, 120)
    edges = cv2.bitwise_and(edges, edges, mask=(bg * 255))

    angles, lengths = _line_angles(edges)
    line_n = int(len(angles))
    if line_n == 0:
        return {
            "angle_deg": 0.0,
            "rot_conf": 0.0,
            "rot_ok": False,
            "rot_fail_reason": "no_lines",
            "rot_line_count": 0,
        }

    hist, _ = np.histogram(angles, bins=180, range=(0.0, 180.0), weights=lengths)
    peak_primary = int(np.argmax(hist))
    peak_primary_value = float(hist[peak_primary])

    hist_secondary = hist.copy()
    for d in range(-8, 9):
        hist_secondary[(peak_primary + d) % 180] = 0.0
    peak_secondary = int(np.argmax(hist_secondary))
    peak_secondary_value = float(hist_secondary[peak_secondary])
    peak_orthogonal = float(hist[(peak_primary + 90) % 180])

    mods = np.mod(angles, 90.0)
    theta = mods * (2.0 * np.pi / 90.0)
    cx = float(np.sum(lengths * np.cos(theta)))
    cy = float(np.sum(lengths * np.sin(theta)))
    rot_mod90 = float((np.degrees(np.arctan2(cy, cx)) * (90.0 / 360.0)) % 90.0)

    peak_ratio_score = peak_primary_value / (peak_primary_value + peak_secondary_value + 1e-6)
    line_score = min(1.0, line_n / 60.0)
    spread = float(np.sqrt(np.average((mods - np.average(mods, weights=lengths)) ** 2, weights=lengths)))
    spread_score = float(np.clip(1.0 - spread / 20.0, 0.0, 1.0))
    ortho_score = float(np.clip(peak_orthogonal / (peak_primary_value + 1e-6), 0.0, 1.0))
    conf = float(np.clip(0.35 * peak_ratio_score + 0.25 * line_score + 0.20 * spread_score + 0.20 * ortho_score, 0.0, 1.0))

    reasons = []
    if line_n < cfg.rot_line_min:
        reasons.append("line_count_low")
    if conf < cfg.rot_conf_min:
        reasons.append("confidence_low")

    return {
        "angle_deg": -rot_mod90,
        "rot_conf": conf,
        "rot_ok": len(reasons) == 0,
        "rot_fail_reason": "|".join(reasons),
        "rot_line_count": line_n,
    }

def rotate_rgb(rgb: np.ndarray, angle_deg: float, pad_value: int) -> np.ndarray:
    if abs(angle_deg) < 1e-6:
        return rgb
    h, w = rgb.shape[:2]
    matrix = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle_deg, 1.0)
    return cv2.warpAffine(
        rgb,
        matrix,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(pad_value, pad_value, pad_value),
    )


class CheckerboardTopNormalizer:
    def __init__(self, cfg: Optional[CheckerboardTopNormConfig] = None) -> None:
        self.cfg = cfg or CheckerboardTopNormConfig()
        self._angle_cache: Dict[str, Optional[float]] = {}

    def normalize(self, path: str | Path, image: Image.Image, debug_save_path: Optional[Path] = None) -> Image.Image:
            """
            이미지를 정규화(회전 보정)합니다.
            debug_save_path가 전달되면 중간 분석 과정을 시각화하여 저장합니다.
            """
            if not self.cfg.enabled:
                return image

            key = str(Path(path).expanduser().resolve())
            
            # 1. 각도 계산 및 캐싱 (이미 계산된 적이 없다면 실행)
            if key not in self._angle_cache:
                rgb = np.asarray(image.convert("RGB"))
                info = estimate_top_rotation(rgb, self.cfg)
                
                # 성공 여부에 따라 각도 저장 (실패 시 None)
                self._angle_cache[key] = float(info["angle_deg"]) if bool(info["rot_ok"]) else None

                # [핵심 수정] 파라미터로 받은 debug_save_path가 있을 때만 시각화 함수 호출
                if debug_save_path:
                    self._save_debug_plot(rgb, info, debug_save_path)

            # 2. 캐시된 각도 가져오기
            angle = self._angle_cache[key]
            
            # 보정할 각도가 없거나(None), 실패한 경우 원본 반환
            if angle is None:
                return image

            # 3. 실제 회전 적용
            rgb = np.asarray(image.convert("RGB"))
            rotated = rotate_rgb(rgb, angle_deg=angle, pad_value=self.cfg.pad_value)
            
            return Image.fromarray(rotated)
    
    def _save_debug_plot(self, rgb: np.ndarray, info: dict, save_path: Path):
        # 중간 과정 재현 (시각화용)
        fg_mask = _estimate_mask(rgb)
        bg_mask = (1 - fg_mask).astype(np.uint8)
        if int(bg_mask.sum()) < int(0.03 * rgb.shape[0] * rgb.shape[1]):
            bg_mask = _ring_mask(rgb.shape[0], rgb.shape[1], self.cfg.ring_ratio)
        
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        edges = cv2.Canny(gray, 40, 120) # 단순화를 위해 바로 Canny 적용 가능
        masked_edges = cv2.bitwise_and(edges, edges, mask=(bg_mask * 255))
        
        line_img = rgb.copy()
        lines = cv2.HoughLinesP(masked_edges, 1, np.pi / 180.0, threshold=30, minLineLength=24, maxLineGap=6)
        if lines is not None:
            for line in lines[:100]:
                x1, y1, x2, y2 = line[0]
                cv2.line(line_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        rotated = rotate_rgb(rgb, info["angle_deg"], self.cfg.pad_value)

        # 플롯 생성
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        titles = ["Original", "Edges", "Lines", "Result"]
        imgs = [rgb, masked_edges, line_img, rotated]
        
        for ax, img, title in zip(axes, imgs, titles):
            ax.imshow(img, cmap='gray' if len(img.shape) == 2 else None)
            ax.set_title(title)
            ax.axis('off')
        
        fig.suptitle(f"Angle: {info['angle_deg']:.2f}° | Conf: {info['rot_conf']:.2f}", fontsize=15)
        
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight')
        plt.close(fig)

## 4-2. VideoMAE & CheckerboardTopNormalization 을 포함한 커스텀 클래스 + 데이터 로더 위한 함수 정의

In [ ]:
class MultiViewDataset(Dataset):
    def __init__(
        self,
        df,
        root_dir,
        transform=None, # 이 transform이 Compose라면 내부 시점별 분기가 필요할 수 있음
        is_test=False,
        feature_dir=None,
        checkerboard_top_normalize: bool = True
    ):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}
        self.feature_dir = Path(feature_dir) if feature_dir else None
        
        # 체커보드 정규화기 설정
        self.top_normalizer = CheckerboardTopNormalizer(
            CheckerboardTopNormConfig(enabled=True)
        ) if checkerboard_top_normalize else None

    def __len__(self):
        return len(self.df)

    def _load_img(self, sid: str, view: str, sample_dir: str) -> Image.Image:
        # 비디오 증강 데이터 여부 확인 (ID prefix 사용)
        is_augv = sid.startswith('AUGV_')
        
        # 경로 설정
        path = Path(sample_dir) / sid / f"{view}.png"
        image = Image.open(path).convert("RGB")
        
        # [핵심 수정] Top View 정규화 로직
        # 비디오 증강 데이터(AUGV_)가 아닐 때만 체커보드 정규화 수행
        if view == "top" and self.top_normalizer is not None and not is_augv:
            debug_path = None
            # 학습/검증 중 앞부분만 디버깅 이미지 저장
            if not self.is_test and len(self.top_normalizer._angle_cache) < 20:
                folder = "train" if "TRAIN" in sid else "dev"
                debug_path = Path(f"./debug_plots/{folder}/{sid}_top.png")

            image = self.top_normalizer.normalize(path, image, debug_save_path=debug_path)
            
        return image

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sid = str(row['id'])
        # sample_dir이 row에 있으면 사용, 없으면 기본 root_dir 사용
        sample_dir = row['sample_dir'] if 'sample_dir' in row else self.root_dir

        views = []
        for name in ['front', 'top']:
            # 정규화 로직이 포함된 이미지 로드 호출
            image = self._load_img(sid, name, sample_dir)
            
            # transform 적용
            if self.transform:
                # 만약 transform이 시점별로 다르다면(Compose 내부 분기 등) 여기서 처리
                image = self.transform(image)
            
            views.append(image)

        # VideoMAE feature 로드 (기존 로직 유지)
        feature = torch.zeros(768)
        if self.feature_dir is not None:
            feat_path = self.feature_dir / f'{sid}.npy'
            if feat_path.exists():
                feature = torch.from_numpy(np.load(feat_path)).float()
            elif sid.startswith('AUGV_'):
                # 증강 데이터인데 피처가 없다면? 
                # 원본 비디오의 피처를 찾거나(복잡), 그냥 0으로 유지
                pass

        if self.is_test:
            return views, feature

        label = self.label_map[row['label']]
        return views, label, feature

In [ ]:
def _extract_frame_by_sec(cap, sec, fps, frame_count):
    # 매 프레임에 해당하는 장면을 가져오는 함수
    frame_idx = int(max(0, min(frame_count - 1, round(sec * fps))))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _extract_last_frame(cap, frame_count):
    # 가장 마지막 프레임을 가져오는 함수
    last_idx = max(0, frame_count - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, last_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _video_aug_cache_signature(cfg):
    # VIDEO_AUG에 해당하는 CFG만 가져오는 함수
    keys = [
        'SEED',
        'UNSTABLE_START_MIN_SEC',
        'UNSTABLE_START_MAX_SEC',
        'UNSTABLE_FRAMES_MIN',
        'UNSTABLE_FRAMES_MAX',
        'STABLE_END_MIN_SEC',
        'STABLE_END_MAX_SEC',
        'STABLE_FRAMES_PER_VIDEO',
    ]
    return {k: cfg.get(k) for k in keys}

def detect_collapse_start(cap, fps, frame_count, motion_threshold=0.02):
    """
    프레임 간 픽셀 변화량으로 붕괴 시작 시점 탐지
    """
    prev_frame = None
    collapse_sec = None

    # 전체 영상을 일정 간격으로 샘플링
    sample_interval = max(1, int(fps * 0.2))  # 0.2초 간격

    for frame_idx in range(0, frame_count, sample_interval):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(float) / 255.0

        if prev_frame is not None:
            # 프레임 간 변화량 계산
            diff = np.mean(np.abs(gray - prev_frame))
            if diff > motion_threshold:
                collapse_sec = frame_idx / fps
                break  # 첫 번째 큰 변화 시점

        prev_frame = gray

    return collapse_sec  # None이면 붕괴 미감지

def build_video_augmented_df(train_df, data_dir, cfg):
    """
    train의 simulation.mp4에서 프레임 추출 (train의 경우 )
    stable : 그냥 계속 동일한 프레임들이 나올테니, 그냥 마지막 프레임 뽑음
    unstable : 0~10초 내에 구조물이 무너지는 순간이 있을텐데, 이 변화량이 큰 순간을 포착
    """
    train_root = data_dir / 'train'
    aug_root = data_dir / 'train_video_aug'
    aug_root.mkdir(parents=True, exist_ok=True)

    cache_csv = aug_root / 'aug_df.csv'
    cache_meta = aug_root / 'cache_meta.json'
    cache_sig = _video_aug_cache_signature(cfg)

    if cfg.get('VIDEO_AUG_CACHE', True) and cache_csv.exists() and cache_meta.exists():
        try:
            meta = json.loads(cache_meta.read_text())
            if meta.get('signature') == cache_sig:
                cached_df = pd.read_csv(cache_csv)
                if not cached_df.empty:
                    cached_df['sample_dir'] = str(aug_root)
                    print(f'video aug cache hit: {len(cached_df)} samples from {cache_csv}')
                    return cached_df
        except Exception as e:
            print(f'video aug cache read failed. rebuild cache. ({e})')

    # 캐시 미스 시 기존 AUGV_* 폴더만 정리 후 재생성
    for p in aug_root.glob('AUGV_*'):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)

    rng = np.random.default_rng(cfg['SEED'])
    stable_rows = []
    unstable_rows = []
    saved_idx = 0

    def save_aug(img, label):
        nonlocal saved_idx
        aug_id = f'AUGV_{saved_idx:07d}'
        out_dir = aug_root / aug_id
        out_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img).save(out_dir / 'front.png')
        Image.fromarray(img).save(out_dir / 'top.png')
        row = {'id': aug_id, 'label': label, 'sample_dir': str(aug_root)}
        saved_idx += 1
        return row

    # 1) stable 증강: 영상의 마지막 프레임 1장씩 사용
    all_ids = train_df['id'].tolist()
    for sample_id in tqdm(all_ids, desc='Video aug stable(last-frame)', dynamic_ncols=True, ascii=True):
        video_path = train_root / sample_id / 'simulation.mp4'
        if not video_path.exists():
            continue

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            cap.release()
            continue

        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if frame_count <= 0:
            cap.release()
            continue

        img = _extract_last_frame(cap, frame_count)
        cap.release()
        if img is None:
            continue

        stable_rows.append(save_aug(img, 'stable'))

    # 2) unstable 증강: 실제로 해당 구조물이 붕괴하기 시작하는 시점을 탐지
    unstable_ids = train_df.loc[train_df['label'] == 'unstable', 'id'].tolist()
    for sample_id in tqdm(unstable_ids, desc='Video aug unstable(collapse-detect)', dynamic_ncols=True, ascii=True):
        video_path = train_root / sample_id / 'simulation.mp4'
        if not video_path.exists():
            continue
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # ← 핵심 변경: 실제 붕괴 시작 시점 탐지
        collapse_sec = detect_collapse_start(cap, fps, frame_count)

        if collapse_sec is None:
            # 탐지 실패 시 기존 방식 fallback
            low  = cfg['UNSTABLE_START_MIN_SEC']
            high = cfg['UNSTABLE_START_MAX_SEC']
        else:
            # 탐지 성공 시 붕괴 시점 기준으로 샘플링 구간 설정
            low  = max(0.0, collapse_sec - 0.3)   # 붕괴 0.3초 전
            high = min(collapse_sec + 0.5,          # 붕괴 0.5초 후
                       frame_count / fps)

        n_unstable = int(rng.integers(
            cfg['UNSTABLE_FRAMES_MIN'],
            cfg['UNSTABLE_FRAMES_MAX'] + 1
        ))
        unstable_secs = rng.uniform(low, high, size=n_unstable)

        for sec in unstable_secs:
            img = _extract_frame_by_sec(cap, sec, fps, frame_count)
            if img is None:
                continue
            unstable_rows.append(save_aug(img, 'unstable'))

        cap.release()

    stable_df = pd.DataFrame(stable_rows)
    unstable_df = pd.DataFrame(unstable_rows)

    if stable_df.empty or unstable_df.empty:
        print('video aug warning: stable/unstable 중 하나가 비어 비율 매칭 불가')
        return pd.DataFrame(columns=['id', 'label', 'sample_dir'])

    # 3) stable 개수에 맞춰 unstable 개수 정렬
    target_unstable = len(stable_df)
    if len(unstable_df) >= target_unstable:
        unstable_bal = unstable_df.sample(n=target_unstable, random_state=cfg['SEED'])
    else:
        unstable_bal = unstable_df.sample(n=target_unstable, replace=True, random_state=cfg['SEED'])

    aug_df = pd.concat([stable_df, unstable_bal], ignore_index=True)
    aug_df = aug_df.sample(frac=1.0, random_state=cfg['SEED']).reset_index(drop=True)

    # 캐시 저장
    if cfg.get('VIDEO_AUG_CACHE', True):
        aug_df.to_csv(cache_csv, index=False)
        cache_meta.write_text(json.dumps({'signature': cache_sig}, ensure_ascii=False, indent=2))
        print(f'video aug cache saved: {cache_csv}')

    print(f'video aug stable(last-frame): {len(stable_df)}')
    print(f'video aug unstable(sampled): {len(unstable_bal)}')
    return aug_df

In [ ]:
train_transform, test_transform = build_default_transforms(CFG['IMG_SIZE'])

# 원본 학습 데이터(기본 1:1)
train_df_copy = train_df.copy()
train_df_copy['sample_dir'] = str(DATA_DIR / 'train')

# 비디오 프레임 기반 증강 데이터 생성
if CFG.get('VIDEO_AUG_ENABLE', False):
    aug_df = build_video_augmented_df(train_df, DATA_DIR, CFG)
    if len(aug_df) > 0:
        train_df_copy = pd.concat([train_df_copy, aug_df], ignore_index=True)
        print(f'video aug added: {len(aug_df)} samples')
    else:
        print('video aug added: 0 samples (check video availability)')

# 최종 학습 비율 확인
print('Final train class ratio:')
print(train_df_copy['label'].value_counts())

In [ ]:
val_df_for_eval = val_df.copy()
val_df_for_eval['sample_dir'] = str(DATA_DIR / 'dev')

FEATURE_DIR = DATA_DIR / 'videomae_features'

# 1. 학습/검증 세트 준비
train_dataset = MultiViewDataset(train_df_copy, str(DATA_DIR / 'train'), train_transform, is_test=False, feature_dir=FEATURE_DIR)
val_dataset = MultiViewDataset(val_df_for_eval, str(DATA_DIR / 'dev'), test_transform, is_test=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=True,
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=(device.type == 'cuda'), # CPU RAM -> GPU VRAM 전송 시 더 빠른 전송 경로 사용
    worker_init_fn=seed_worker,
    generator=make_generator(CFG['SEED'])
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=(device.type == 'cuda'), # CPU RAM -> GPU VRAM 전송 시 더 빠른 전송 경로 사용
    worker_init_fn=seed_worker,
    generator=make_generator(CFG['SEED'] + 1)
)

# 2. 테스트 세트 준비
test_df = pd.read_csv(DATA_DIR / 'sample_submission.csv', encoding='utf-8-sig')
test_df_for_infer = test_df.copy()
test_df_for_infer['sample_dir'] = str(DATA_DIR / 'test')

test_dataset = MultiViewDataset(test_df_for_infer, str(DATA_DIR / 'test'), test_transform, is_test=True)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=(device.type == 'cuda'),
    worker_init_fn=seed_worker,
    generator=make_generator(CFG['SEED'] + 2)
)

In [ ]:
sample = train_dataset[0]

# 현재 Dataset의 return이 (views, label, feature) 구조인지 확인
views, label, feature = sample

print("--- Data Sample Check ---")
print(f"1. Views (Image List): {len(views)} images")
print(f"   - Front View Shape: {views[0].shape}") # Tensor일 경우 [C, H, W]
print(f"   - Top View Shape:   {views[1].shape}")

print(f"2. Label: {label} (0: stable, 1: unstable)")

print(f"3. Feature (VideoMAE): {feature.shape}")
print(f"   - Sample Values: {feature[:5]}...") # 앞부분 5개 값만 확인

## 5. 모델 정의 (Multi-View Model Comparison)


In [ ]:
from models import (
    EMAConfig,
    ModelEMA,
    MultiViewFeatureFusion,
    MultiViewFeatureFusionConfig,
    MultiViewCrossAttention,
    MultiViewCrossAttentionConfig,
    MultiViewBidirectionalCrossAttention,
    MultiViewBidirectionalCrossAttentionConfig,
    MultiViewDualEncoderBidirectionalCrossAttention,
    MultiViewDualEncoderBidirectionalCrossAttentionConfig,
)

EXPERIMENT_NAME = 'model_comparison_regularization_format'
MODEL_COMP_DIR = (Path.cwd() / '../outputs/model_comparison' / EXPERIMENT_NAME).resolve()
WEIGHT_DIR = MODEL_COMP_DIR / 'weights'
HISTORY_DIR = MODEL_COMP_DIR / 'history'
PLOT_DIR = MODEL_COMP_DIR / 'plots'
DEV_PRED_DIR = MODEL_COMP_DIR / 'dev_predictions'
SUBMISSION_DIR = MODEL_COMP_DIR / 'submissions'
REPORT_PATH = MODEL_COMP_DIR / 'portfolio_report.md'
SUMMARY_PATH = MODEL_COMP_DIR / 'summary.csv'

for path in [WEIGHT_DIR, HISTORY_DIR, PLOT_DIR, DEV_PRED_DIR, SUBMISSION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

EMA_CONFIG = EMAConfig(decay=CFG['EMA_DECAY'])
print(f"Output: {MODEL_COMP_DIR}")


## 6. 학습 및 검증 루프 구현

In [ ]:
from torch.cuda.amp import autocast, GradScaler

def _binary_metrics(probs, labels):
    probs = np.asarray(probs, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    eps = 1e-15
    p = np.clip(probs, eps, 1 - eps)
    logloss_score = -np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))
    acc_score = np.mean((probs > 0.5) == labels)
    try:
        auc_score = roc_auc_score(labels, probs)
    except ValueError:
        auc_score = np.nan
    return float(logloss_score), float(acc_score), float(auc_score)


def train_one_epoch(model, loader, criterion, optimizer, device, ema=None, scaler=None, epoch=0):
    model.train()
    epoch_total_loss = 0.0
    use_amp = bool(CFG.get('USE_AMP', True) and device.type == 'cuda')
    pbar = tqdm(enumerate(loader), total=len(loader), desc=f"Epoch {epoch}", dynamic_ncols=True, ascii=True)

    for i, batch in pbar:
        views, labels = batch[0], batch[1]
        views = [v.to(device, non_blocking=True) for v in views]
        labels = labels.to(device, non_blocking=True).float()

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=use_amp):
            outputs = model(views).view(-1)
            loss = criterion(outputs, labels)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        if ema is not None:
            ema.update(model)

        epoch_total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_total = epoch_total_loss / max(len(loader), 1)
    print()
    print(f">> [Epoch {epoch} Summary] | Train Loss: {avg_total:.6f} | Learning Rate: {optimizer.param_groups[0]['lr']:.8f}")

    if CFG.get('USE_WANDB', False):
        wandb.log({'train/epoch_total_loss': avg_total, 'epoch': epoch})

    return avg_total


def validate(model, loader, criterion, device, return_probs=False):
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validation", dynamic_ncols=True, ascii=True):
            views, labels = batch[0], batch[1]
            views = [v.to(device, non_blocking=True) for v in views]
            labels = labels.to(device, non_blocking=True).float()

            outputs = model(views).view(-1)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            all_preds.append(torch.sigmoid(outputs).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs = np.concatenate(all_preds, axis=0).astype(np.float64)
    all_labels = np.concatenate(all_labels, axis=0).astype(np.float64)
    logloss_score, acc_score, auc_score = _binary_metrics(all_probs, all_labels)
    avg_bce_loss = val_loss / max(len(loader), 1)

    if return_probs:
        return logloss_score, acc_score, auc_score, avg_bce_loss, all_probs, all_labels
    return logloss_score, acc_score, auc_score


# -------------------------
# TTA helpers
# -------------------------
def _center_crop_and_resize(x, crop_ratio=0.95):
    b, c, h, w = x.shape
    ch, cw = int(h * crop_ratio), int(w * crop_ratio)
    y1 = (h - ch) // 2
    x1 = (w - cw) // 2
    cropped = x[:, :, y1:y1 + ch, x1:x1 + cw]
    return F.interpolate(cropped, size=(h, w), mode='bilinear', align_corners=False)


def apply_tta_to_views(views, tta_name):
    if tta_name == 'none':
        return views
    if tta_name == 'hflip':
        return [torch.flip(v, dims=[3]) for v in views]
    if tta_name == 'crop95':
        return [_center_crop_and_resize(v, crop_ratio=0.95) for v in views]
    raise ValueError(f'Unknown TTA: {tta_name}')


def predict_probs_with_tta(model, loader, device, tta_names=None, has_labels=False, desc='Inference TTA'):
    if tta_names is None:
        tta_names = ['none']

    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc, dynamic_ncols=True, ascii=True):
            if has_labels:
                views, labels = batch[0], batch[1]
                labels = labels.to(device).float()
                all_labels.extend(labels.cpu().numpy())
            else:
                views = batch[0]

            views = [v.to(device) for v in views]

            probs_sum = None
            for tta_name in tta_names:
                tta_views = apply_tta_to_views(views, tta_name)
                logits = model(tta_views).view(-1)
                probs = torch.sigmoid(logits)
                probs_sum = probs if probs_sum is None else (probs_sum + probs)

            probs_avg = probs_sum / len(tta_names)
            all_probs.extend(probs_avg.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    if has_labels:
        return all_probs, np.array(all_labels, dtype=np.float64)
    return all_probs


def evaluate_tta_on_dev(model, loader, device, tta_candidates):
    rows = []
    for tta_names in tta_candidates:
        probs, labels = predict_probs_with_tta(
            model, loader, device, tta_names=tta_names, has_labels=True,
            desc=f'Dev TTA {tta_names}'
        )
        logloss_score, acc_score, auc_score = _binary_metrics(probs, labels)
        rows.append({
            'tta_names': tta_names,
            'n_tta': len(tta_names),
            'val_logloss': float(logloss_score),
            'val_acc': float(acc_score),
            'val_auc': float(auc_score),
        })

    return pd.DataFrame(rows).sort_values('val_logloss', ascending=True).reset_index(drop=True)


In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    name: str
    model_cls: type
    config_cls: type
    config_kwargs: dict


def safe_name(name: str) -> str:
    return name.replace('/', '_').replace('.', '_').replace(' ', '_')


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_model_specs():
    backbone_name = CFG['BACKBONE_NAME']
    pretrained = CFG['PRETRAINED']
    common = {'backbone_name': backbone_name, 'pretrained': pretrained}
    front_backbone = CFG.get('FRONT_BACKBONE_NAME') or backbone_name
    top_backbone = CFG.get('TOP_BACKBONE_NAME') or front_backbone

    return {
        'feature_fusion': ModelSpec(
            name='feature_fusion',
            model_cls=MultiViewFeatureFusion,
            config_cls=MultiViewFeatureFusionConfig,
            config_kwargs={**common, 'dropout': CFG['CLASSIFIER_DROPOUT']},
        ),
        'cross_attention': ModelSpec(
            name='cross_attention',
            model_cls=MultiViewCrossAttention,
            config_cls=MultiViewCrossAttentionConfig,
            config_kwargs={**common, 'attn_dim': CFG['ATTN_DIM'], 'num_heads': CFG['NUM_HEADS']},
        ),
        'bidirectional_cross_attention': ModelSpec(
            name='bidirectional_cross_attention',
            model_cls=MultiViewBidirectionalCrossAttention,
            config_cls=MultiViewBidirectionalCrossAttentionConfig,
            config_kwargs={
                **common,
                'attn_dim': CFG['ATTN_DIM'],
                'num_heads': CFG['NUM_HEADS'],
                'num_layers': CFG['NUM_LAYERS'],
                'dropout': CFG['ATTN_DROPOUT'],
                'classifier_dropout': CFG['CLASSIFIER_DROPOUT'],
            },
        ),
        'dual_encoder_bidirectional_cross_attention': ModelSpec(
            name='dual_encoder_bidirectional_cross_attention',
            model_cls=MultiViewDualEncoderBidirectionalCrossAttention,
            config_cls=MultiViewDualEncoderBidirectionalCrossAttentionConfig,
            config_kwargs={
                'front_backbone_name': front_backbone,
                'top_backbone_name': top_backbone,
                'pretrained': pretrained,
                'attn_dim': CFG['ATTN_DIM'],
                'num_heads': CFG['NUM_HEADS'],
                'num_layers': CFG['NUM_LAYERS'],
                'dropout': CFG['ATTN_DROPOUT'],
                'classifier_dropout': CFG['CLASSIFIER_DROPOUT'],
            },
        ),
    }


def make_model(spec: ModelSpec):
    config = spec.config_cls(**spec.config_kwargs)
    model = spec.model_cls(config)
    return model, config


MODEL_SPECS = build_model_specs()
MODEL_SPECS


In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

def _save_history_plot(history_df, plot_path, title):
    if not CFG.get('PORTFOLIO_ARTIFACTS', True) or history_df.empty:
        return None

    fig, ax_loss = plt.subplots(figsize=(9, 5))
    ax_metric = ax_loss.twinx()
    ax_loss.plot(history_df['epoch'], history_df['train_loss'], marker='o', label='train loss')
    ax_loss.plot(history_df['epoch'], history_df['val_logloss'], marker='s', label='dev logloss')
    ax_metric.plot(history_df['epoch'], history_df['val_acc'], marker='^', color='green', label='dev acc')
    if 'val_auc' in history_df.columns:
        ax_metric.plot(history_df['epoch'], history_df['val_auc'], marker='D', color='purple', label='dev auc')

    best_idx = history_df['val_logloss'].idxmin()
    best_row = history_df.loc[best_idx]
    ax_loss.axvline(best_row['epoch'], linestyle='--', color='gray', alpha=0.6)
    ax_loss.scatter([best_row['epoch']], [best_row['val_logloss']], s=90, color='orange', zorder=5)

    ax_loss.set_title(title)
    ax_loss.set_xlabel('epoch')
    ax_loss.set_ylabel('loss / logloss')
    ax_metric.set_ylabel('accuracy / auc')
    ax_loss.grid(True, linestyle='--', alpha=0.25)

    lines1, labels1 = ax_loss.get_legend_handles_labels()
    lines2, labels2 = ax_metric.get_legend_handles_labels()
    ax_loss.legend(lines1 + lines2, labels1 + labels2, loc='best')
    fig.tight_layout()
    plot_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(plot_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return str(plot_path)


def _save_summary_plot(summary_df):
    if not CFG.get('PORTFOLIO_ARTIFACTS', True) or summary_df.empty:
        return None
    plot_df = summary_df.sort_values('best_val_logloss').copy()
    plot_df['label'] = plot_df['model_name'].astype(str) + ' / ep' + plot_df['target_epochs'].astype(str)

    fig_h = max(4.5, 0.45 * len(plot_df) + 1.5)
    fig, ax = plt.subplots(figsize=(10, fig_h))
    ax.barh(plot_df['label'], plot_df['best_val_logloss'])
    ax.invert_yaxis()
    ax.set_xlabel('best dev logloss')
    ax.set_title('src/models architecture comparison')
    ax.grid(axis='x', linestyle='--', alpha=0.25)
    for y, value in enumerate(plot_df['best_val_logloss']):
        ax.text(value, y, f' {value:.5f}', va='center')
    fig.tight_layout()
    plot_path = PLOT_DIR / 'summary_comparison.png'
    fig.savefig(plot_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return str(plot_path)


def _write_portfolio_report(summary_df):
    if not CFG.get('PORTFOLIO_ARTIFACTS', True) or summary_df.empty:
        return None
    ranked = summary_df.sort_values('best_val_logloss').reset_index(drop=True)
    best = ranked.iloc[0]

    display_cols = [
        'model_name', 'target_epochs', 'best_epoch', 'best_val_logloss',
        'best_val_acc', 'best_val_auc', 'best_tta_val_logloss', 'trainable_params'
    ]
    display_cols = [c for c in display_cols if c in ranked.columns]
    table_df = ranked[display_cols].copy()

    def _markdown_table(frame):
        header = '| ' + ' | '.join(frame.columns) + ' |'
        sep = '| ' + ' | '.join(['---'] * len(frame.columns)) + ' |'
        body = []
        for _, row in frame.iterrows():
            values = []
            for value in row.tolist():
                if isinstance(value, float):
                    values.append(f'{value:.6f}')
                else:
                    values.append(str(value))
            body.append('| ' + ' | '.join(values) + ' |')
        return '\n'.join([header, sep, *body])

    lines = [
        '# Structure Stability Model Comparison',
        '',
        'Regularization notebook format을 유지해 `src/models` 아키텍처를 비교한 결과입니다.',
        'WandB 대신 포트폴리오에 바로 넣기 쉬운 정적 산출물(CSV/PNG/Markdown)을 남깁니다.',
        '',
        '## Best Run',
        '',
        f"- Model: `{best['model_name']}`",
        f"- Target epochs: `{best['target_epochs']}`",
        f"- Best epoch: `{best['best_epoch']}`",
        f"- Best dev logloss: `{best['best_val_logloss']:.6f}`",
        f"- Best dev accuracy: `{best['best_val_acc']:.6f}`",
        f"- Best dev AUC: `{best['best_val_auc']:.6f}`",
        '',
        '## Ranked Results',
        '',
        _markdown_table(table_df),
        '',
        '## Artifacts',
        '',
        f"- Summary CSV: `{SUMMARY_PATH}`",
        f"- Summary plot: `{PLOT_DIR / 'summary_comparison.png'}`",
        f"- Learning curves: `{PLOT_DIR}`",
        f"- Dev predictions: `{DEV_PRED_DIR}`",
        f"- Submissions: `{SUBMISSION_DIR}`",
        '',
    ]
    REPORT_PATH.write_text('\n'.join(lines), encoding='utf-8')
    return str(REPORT_PATH)


def train_single_model(model_name: str):
    spec = MODEL_SPECS[model_name]
    sweep_results = []
    epochs_list = CFG['EPOCHS']

    for target_epochs in epochs_list:
        run_name = f"{safe_name(model_name)}_{safe_name(CFG['BACKBONE_NAME'])}_ep{target_epochs}"
        best_model_path = WEIGHT_DIR / f"best_model_{run_name}.pt"
        history_path = HISTORY_DIR / f"history_{run_name}.csv"
        plot_path = PLOT_DIR / f"learning_curve_{run_name}.png"

        if best_model_path.exists():
            print()
            print(f"[Skip] 이미 학습된 모델이 존재합니다: {best_model_path.name}")
            checkpoint = torch.load(best_model_path, map_location='cpu', weights_only=False)
            row = checkpoint.get('summary', {
                'model_name': model_name,
                'target_epochs': target_epochs,
                'best_epoch': checkpoint.get('epoch', -1),
                'best_val_logloss': checkpoint.get('val_logloss', np.nan),
                'best_val_acc': checkpoint.get('val_acc', np.nan),
                'best_val_auc': checkpoint.get('val_auc', np.nan),
                'checkpoint_path': str(best_model_path),
                'history_path': str(history_path),
            })
            sweep_results.append(row)
            continue

        print()
        print("=======================================================")
        print(f" Starting Sweep: {model_name} | EPOCHS = {target_epochs}")
        print("=======================================================")
        print()

        if CFG.get('USE_WANDB', False):
            wandb.init(
                project='structure-stability',
                name=run_name,
                group=EXPERIMENT_NAME,
                config=CFG,
                reinit=True,
            )

        model, model_config = make_model(spec)
        model = model.to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'], weight_decay=CFG['WEIGHT_DECAY'])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=target_epochs, eta_min=CFG['MIN_LR']
        )
        ema = ModelEMA(model, EMA_CONFIG)
        scaler = GradScaler(enabled=bool(CFG.get('USE_AMP', True) and device.type == 'cuda'))
        trainable_params = count_trainable_params(model)

        print(model_config)
        print(f"Trainable params: {trainable_params:,}")
        print(f"Regularization -> weight_decay={CFG['WEIGHT_DECAY']}")
        print(f"Scheduler -> CosineAnnealingLR(T_max={target_epochs}, eta_min={CFG['MIN_LR']})")
        print(f"EMA -> decay={CFG['EMA_DECAY']}, use_for_eval={CFG['EMA_USE_FOR_EVAL']}")
        print(f"Early Stopping -> Patience={CFG['PATIENCE']}")

        best_epoch = -1
        early_stop_counter = 0
        best_logloss = float('inf')
        best_acc = np.nan
        best_auc = np.nan
        history = []

        for epoch in range(1, target_epochs + 1):
            avg_train_loss = train_one_epoch(
                model, train_loader, criterion, optimizer, device,
                ema=ema,
                scaler=scaler,
                epoch=epoch,
            )

            eval_model = ema.ema_model if CFG['EMA_USE_FOR_EVAL'] else model
            val_logloss, val_acc, val_auc = validate(eval_model, val_loader, criterion, device)

            if CFG.get('USE_WANDB', False):
                wandb.log({
                    'epoch': epoch,
                    'val/logloss': val_logloss,
                    'val/acc': val_acc,
                    'val/auc': val_auc,
                    'lr': optimizer.param_groups[0]['lr'],
                    'best_logloss': best_logloss,
                    'early_stop_count': early_stop_counter,
                }, step=epoch)

            improved = val_logloss < best_logloss
            if improved:
                best_logloss = val_logloss
                best_acc = val_acc
                best_auc = val_auc
                best_epoch = epoch
                early_stop_counter = 0
                torch.save({
                    'epoch': epoch,
                    'model_name': model_name,
                    'model_config': asdict(model_config),
                    'model_state_dict': model.state_dict(),
                    'ema_state_dict': ema.ema_model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_logloss': val_logloss,
                    'val_acc': val_acc,
                    'val_auc': val_auc,
                    'cfg': CFG,
                }, best_model_path)
                print(f"  -> Best model saved: {best_model_path} (epoch={epoch}, val_logloss={val_logloss:.6f})")
            else:
                early_stop_counter += 1
                print(f"  -> No improvement. EarlyStopping counter: {early_stop_counter}/{CFG['PATIENCE']}")

            scheduler.step()
            current_lr = optimizer.param_groups[0]['lr']
            history.append({
                'model_name': model_name,
                'target_epochs': target_epochs,
                'epoch': epoch,
                'train_loss': avg_train_loss,
                'val_logloss': val_logloss,
                'val_acc': val_acc,
                'val_auc': val_auc,
                'lr': current_lr,
                'improved': improved,
                'early_stop_counter': early_stop_counter,
            })

            print(f"Epoch [{epoch}]")
            print(f"  - LR: {current_lr:.8f}")
            print(f"  - Train Loss: {avg_train_loss:.4f}")
            print(f"  - Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")

            if early_stop_counter >= CFG['PATIENCE']:
                print()
                print(f"! [Early Stopping] Training stopped at epoch {epoch}. Best Epoch was {best_epoch}.")
                break

        if CFG.get('USE_WANDB', False):
            wandb.finish()

        history_df = pd.DataFrame(history)
        history_df.to_csv(history_path, index=False)
        learning_curve_path = _save_history_plot(history_df, plot_path, title=run_name)

        checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
        summary = {
            'model_name': model_name,
            'target_epochs': target_epochs,
            'best_epoch': checkpoint['epoch'],
            'best_val_logloss': checkpoint['val_logloss'],
            'best_val_acc': checkpoint['val_acc'],
            'best_val_auc': checkpoint.get('val_auc', np.nan),
            'trainable_params': trainable_params,
            'checkpoint_path': str(best_model_path),
            'history_path': str(history_path),
            'learning_curve_path': learning_curve_path,
        }
        checkpoint['summary'] = summary
        torch.save(checkpoint, best_model_path)
        sweep_results.append(summary)

    return sweep_results


In [ ]:
results = []

for model_name in CFG['MODEL_NAMES']:
    result_list = train_single_model(model_name)
    results.extend(result_list)

summary_df = pd.DataFrame(results).sort_values('best_val_logloss', ascending=True).reset_index(drop=True)
summary_df.to_csv(SUMMARY_PATH, index=False)
summary_plot_path = _save_summary_plot(summary_df)
report_path = _write_portfolio_report(summary_df)

print(f"Summary saved: {SUMMARY_PATH}")
print(f"Summary plot: {summary_plot_path}")
print(f"Portfolio report: {report_path}")
display(summary_df)

best_row = summary_df.iloc[0]
best_model_name = best_row['model_name']
best_checkpoint_path = Path(best_row['checkpoint_path'])
print(f"Best model: {best_model_name} | checkpoint={best_checkpoint_path}")


## 7. 검증셋 오답 확인

비교 결과 1위 모델을 기준으로 dev 오답을 확인합니다.


In [ ]:
def load_model_from_checkpoint(model_name, checkpoint_path, use_ema=True):
    spec = MODEL_SPECS[model_name]
    model, model_config = make_model(spec)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_key = 'ema_state_dict' if use_ema and checkpoint.get('ema_state_dict') is not None else 'model_state_dict'
    model.load_state_dict(checkpoint[state_key])
    model = model.to(device)
    model.eval()
    return model, checkpoint

best_model, best_checkpoint = load_model_from_checkpoint(
    best_model_name,
    best_checkpoint_path,
    use_ema=CFG['EMA_USE_FOR_EVAL'],
)

# 오답 확인은 기본 TTA 후보 중 가장 단순한 none으로 먼저 봅니다.
best_tta_names = ['none']
val_probs, _ = predict_probs_with_tta(
    best_model, val_loader, device,
    tta_names=best_tta_names,
    has_labels=True,
    desc='Validate Error Analysis (TTA)'
)

val_result = val_df.copy().reset_index(drop=True)
val_result['unstable_prob'] = val_probs
val_result['stable_prob'] = 1.0 - val_probs
val_result['pred_label'] = np.where(val_result['unstable_prob'] > 0.5, 'unstable', 'stable')

mistakes = val_result[val_result['pred_label'] != val_result['label']].copy()
mistakes['pred_confidence'] = np.where(
    mistakes['pred_label'] == 'unstable',
    mistakes['unstable_prob'],
    mistakes['stable_prob']
)
mistakes = mistakes.sort_values('pred_confidence', ascending=False).reset_index(drop=True)

print(f"사용 TTA: {best_tta_names}")
print(f"오답 개수: {len(mistakes)} / {len(val_result)}")
display(mistakes[['id', 'label', 'pred_label', 'unstable_prob', 'stable_prob', 'pred_confidence']].head(20))

best_dev_pred_path = DEV_PRED_DIR / f"dev_predictions_{safe_name(best_model_name)}.csv"
val_result.to_csv(best_dev_pred_path, index=False)
print(f"dev predictions saved: {best_dev_pred_path}")


In [ ]:
import matplotlib.pyplot as plt

TOP_N = 8  # 시각화할 오답 샘플 수
show_df = mistakes.head(TOP_N)

for _, row in show_df.iterrows():
    sample_id = row['id']
    front_path = DATA_DIR / 'dev' / sample_id / 'front.png'
    top_path = DATA_DIR / 'dev' / sample_id / 'top.png'

    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    axes[0].imshow(Image.open(front_path).convert('RGB'))
    axes[0].set_title('front')
    axes[0].axis('off')

    axes[1].imshow(Image.open(top_path).convert('RGB'))
    axes[1].set_title('top')
    axes[1].axis('off')

    fig.suptitle(
        f"id={sample_id} | true={row['label']} | pred={row['pred_label']} | unstable_prob={row['unstable_prob']:.4f}",
        fontsize=10
    )
    plt.tight_layout()
    plt.show()

## 8. 추론 및 제출 파일 생성

비교 결과 1위 모델에서 dev TTA를 고른 뒤, test submission을 생성합니다.


In [ ]:
# 1) dev에서 TTA 조합 성능 비교
candidate_ttas = CFG['TTA_CANDIDATES']
tta_result_df = evaluate_tta_on_dev(best_model, val_loader, device, candidate_ttas)
display(tta_result_df)

best_tta_names = tta_result_df.iloc[0]['tta_names']
print(f"Best TTA on dev: {best_tta_names}")

# summary에 best TTA 결과 반영
if 'summary_df' in globals() and not summary_df.empty:
    best_idx = summary_df['checkpoint_path'].eq(str(best_checkpoint_path))
    if best_idx.any():
        summary_df.loc[best_idx, 'best_tta_names'] = str(best_tta_names)
        summary_df.loc[best_idx, 'best_tta_val_logloss'] = float(tta_result_df.iloc[0]['val_logloss'])
        summary_df.loc[best_idx, 'best_tta_val_acc'] = float(tta_result_df.iloc[0]['val_acc'])
        summary_df.loc[best_idx, 'best_tta_val_auc'] = float(tta_result_df.iloc[0]['val_auc'])
        summary_df.to_csv(SUMMARY_PATH, index=False)
        _save_summary_plot(summary_df)
        _write_portfolio_report(summary_df)

# 2) best TTA로 test 추론
all_probs = predict_probs_with_tta(
    best_model, test_loader, device,
    tta_names=best_tta_names,
    has_labels=False,
    desc='Inference with TTA'
)

# 결과 저장
submission = pd.DataFrame({
    'id': test_df['id'],
    'unstable_prob': all_probs,
    'stable_prob': 1.0 - all_probs
})

submission_path = SUBMISSION_DIR / f"submission_{safe_name(best_model_name)}_best.csv"
submission.to_csv(submission_path, encoding='UTF-8-sig', index=False)
print(f'{submission_path} 저장 완료.')
